# G3b — Robustness checks for the affective-trajectory forecaster

In G3, `traj_only` (a forecaster on the 12-d soft recognizer outputs of clips I–III) beat end-to-end `B1`.
Before the test split is opened, this notebook checks whether that survives four objections:

1. **Feature-quality mismatch.** In G3, training rows got trajectories from *one* fold model while validation
   rows got the *average of 5 fold models*. Here every validation prediction is made once per fold model's
   trajectory and the **predictions** are averaged, so train and eval features come from the same kind of model.
   The old feature-averaged score is still reported as a diagnostic.
2. **Recognizer instability.** Stage 1 trains 3 recognizer seeds per fold. The main trajectory averages them;
   each single-seed trajectory is also run on its own.
3. **Selection protocol.** Every comparison is run under two protocols:
   `inner_dev` (train on 32 episodes, early-stop on 5 held-out training episodes) and
   `val` (train on all 37 episodes, early-stop on validation — the report's protocol, optimistic for everyone).
4. **Which clips matter.** Clip ablation for the trajectory model: III, II+III, I+II+III.

A logistic regression on the same trajectory (C chosen by episode-grouped CV on train only) is included as the
simplest possible forecaster. Test stays locked.

In [ ]:
# ======== CONFIG ========
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv"
OUT_DIR = "/kaggle/working"

SEEDS = [42, 123, 456, 789, 1024]      # forecaster seeds
REC_SEEDS = [42, 123, 456]             # recognizer seeds per fold (stage 1)
N_FOLDS = 5
N_INNER_DEV_SOURCES = 5
REC_EPOCHS, FC_EPOCHS, PATIENCE = 60, 50, 8
REC_BATCH, FC_BATCH = 64, 32
LR, WEIGHT_DECAY = 1e-4, 1e-5
POL_WEIGHT = 0.3
CERT_WEIGHTS = {'1': 1.0, '2': 0.75, '3': 0.5}   # only used for bookkeeping here (no weighted arms)

# (name, arm, clips, trajectory source, selection protocol); clips must be a suffix of (1, 2, 3)
EXPERIMENTS = [
    ("B1",                "B1",   (1, 2, 3), None,  "inner_dev"),
    ("traj_I-III",        "traj", (1, 2, 3), "avg", "inner_dev"),
    ("traj_II-III",       "traj", (2, 3),    "avg", "inner_dev"),
    ("traj_III",          "traj", (3,),      "avg", "inner_dev"),
] + [
    (f"traj_I-III_rec{r}", "traj", (1, 2, 3), r,     "inner_dev") for r in REC_SEEDS
] + [
    ("B1@val",            "B1",   (1, 2, 3), None,  "val"),
    ("traj_I-III@val",    "traj", (1, 2, 3), "avg", "val"),
    ("traj_II-III@val",   "traj", (2, 3),    "avg", "val"),
    ("traj_III@val",      "traj", (3,),      "avg", "val"),
]
EVAL_SPLIT = "val"
UNLOCK_TEST = False

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = os.path.join(DATASET_DIR, "Hi-EF-20260829T071606Z-1-001", "Hi-EF", "annotation.csv")
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
train_all = sp[sp.split == 'train'].reset_index(drop=True)
ev = eval_rows(sp, EVAL_SPLIT, UNLOCK_TEST)

train_sources = sorted(train_all.source_folder.unique())
rng = random.Random(0)
shuffled = train_sources[:]
rng.shuffle(shuffled)
FOLD_OF = {s: i % N_FOLDS for i, s in enumerate(shuffled)}
inner_dev_sources = sorted(random.Random(1).sample(train_sources, N_INNER_DEV_SOURCES))

lab = ann[ann[7].notna()].copy()
lab['ep'] = [c.split('/')[0] for c in lab.index]
lab['y_e'] = lab[7].map(E2I)
lab['y_p'] = lab[5].map(lambda p: P2I.get(p, -1))
lab = lab[lab.y_e.notna()]

for d in (train_all, ev):
    d['w_cert'] = d['clip4'].map(lambda c: CERT_WEIGHTS.get(str(ann.at[c, 8]), 1.0))
    d['unc_B'] = d['clip4'].map(lambda c: str(ann.at[c, 8]))
print(f"train {len(train_all)} | {EVAL_SPLIT} {len(ev)} | folds: "
      f"{[sorted(s for s in train_sources if FOLD_OF[s] == k) for k in range(N_FOLDS)]}")
print(f"forecaster inner-dev episodes: {inner_dev_sources}")

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        h = self.inter(tok + self.clip_pos[:, 3 - n:])   # last n positions: clip III is always position 3
        return self.head(h.mean(1))

## Stage 1 — cross-fitted recognizers (5 folds × 3 seeds)

In [ ]:
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def rec_predict(model, clips, bs=512):
    model.eval()
    pe, pp = [], []
    with torch.no_grad():
        for i in range(0, len(clips), bs):
            idx = torch.tensor([CIDX[c] for c in clips[i:i + bs]], device=DEVICE)
            le, lp = model(gather(idx))
            pe.append(F.softmax(le, -1).cpu()); pp.append(F.softmax(lp, -1).cpu())
    return torch.cat(pe).numpy(), torch.cat(pp).numpy()


def train_recognizer(fit_sources, dev_sources, seed):
    seed_all(seed)
    clips = sorted(lab.index[lab.ep.isin(fit_sources)])
    dev = sorted(set(train_all[train_all.source_folder.isin(dev_sources)].clip3))
    y_e = torch.tensor([int(lab.at[c, 'y_e']) for c in clips], device=DEVICE)
    y_p = torch.tensor([int(lab.at[c, 'y_p']) for c in clips], device=DEVICE)
    cidx = torch.tensor([CIDX[c] for c in clips], device=DEVICE)
    dev_y = np.array([int(lab.at[c, 'y_e']) for c in dev])
    model = ClipRecognizer().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best, best_state, bad = -1, None, 0
    for ep in range(REC_EPOCHS):
        model.train()
        perm = torch.randperm(len(clips), device=DEVICE)
        for i in range(0, len(perm), REC_BATCH):
            j = perm[i:i + REC_BATCH]
            le, lp = model(gather(cidx[j]))
            loss = F.cross_entropy(le, y_e[j])
            if (y_p[j] >= 0).any():
                loss = loss + POL_WEIGHT * F.cross_entropy(lp, y_p[j], ignore_index=-1)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        dev_uar = war_uar(rec_predict(model, dev)[0].argmax(1), dev_y, 7)[1]
        if dev_uar > best:
            best, bad = dev_uar, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best


def rec_vector(pe, pp):
    ent = -(pe * np.log(np.clip(pe, 1e-9, 1))).sum(1, keepdims=True)
    return np.concatenate([pe, pp, pe.max(1, keepdims=True), ent], 1).astype(np.float32)

In [ ]:
ev_ctx = sorted(set(ev[['clip1', 'clip2', 'clip3']].values.ravel()))
OOF = {r: {} for r in REC_SEEDS}                   # clip -> (pe, pp) for training episodes, out-of-fold
EVP = {r: [None] * N_FOLDS for r in REC_SEEDS}     # per fold model: (pe, pp) aligned with ev_ctx
rec_log = []
for k in range(N_FOLDS):
    held = [s for s in train_sources if FOLD_OF[s] == k]
    rest = [s for s in train_sources if FOLD_OF[s] != k]
    rdev = sorted(random.Random(100 + k).sample(rest, 4))
    fit = [s for s in rest if s not in rdev]
    assert not set(lab.index[lab.ep.isin(held)]) & set(lab.index[lab.ep.isin(fit)])
    held_clips = sorted(set(train_all[train_all.source_folder.isin(held)][['clip1', 'clip2', 'clip3']].values.ravel()))
    for r in REC_SEEDS:
        model, dev_uar = train_recognizer(fit, rdev, r + 1000 * k)
        pe, pp = rec_predict(model, held_clips)
        OOF[r].update({c: (pe[i], pp[i]) for i, c in enumerate(held_clips)})
        EVP[r][k] = rec_predict(model, ev_ctx)
        rec_log.append({'fold': k, 'rec_seed': r, 'inner_dev_UAR': dev_uar})
        print(f"fold {k} seed {r}: recognizer inner-dev UAR {dev_uar:.2f}")
        del model
        torch.cuda.empty_cache()


def build_traj(mode):
    """mode 'avg' averages the recognizer seeds' posteriors; an int uses that seed alone.
    Returns (train map, list of per-fold eval maps, feature-averaged eval map)."""
    rs = REC_SEEDS if mode == 'avg' else [mode]
    clips = sorted(OOF[rs[0]])
    pe = np.mean([np.stack([OOF[r][c][0] for c in clips]) for r in rs], 0)
    pp = np.mean([np.stack([OOF[r][c][1] for c in clips]) for r in rs], 0)
    train_map = dict(zip(clips, rec_vector(pe, pp)))
    versions = []
    for k in range(N_FOLDS):
        pe = np.mean([EVP[r][k][0] for r in rs], 0)
        pp = np.mean([EVP[r][k][1] for r in rs], 0)
        versions.append(dict(zip(ev_ctx, rec_vector(pe, pp))))
    pe = np.mean([EVP[r][k][0] for r in rs for k in range(N_FOLDS)], 0)
    pp = np.mean([EVP[r][k][1] for r in rs for k in range(N_FOLDS)], 0)
    return train_map, versions, dict(zip(ev_ctx, rec_vector(pe, pp)))


TRAJ = {m: build_traj(m) for m in ['avg'] + REC_SEEDS}
for m, (tmap, versions, favg) in TRAJ.items():
    tr_u = war_uar(np.stack([tmap[c][:7] for c in train_all.clip3]).argmax(1), train_all.yA, 7)[1]
    ev_u = np.mean([war_uar(np.stack([v[c][:7] for c in ev.clip3]).argmax(1), ev.yA, 7)[1] for v in versions])
    fa_u = war_uar(np.stack([favg[c][:7] for c in ev.clip3]).argmax(1), ev.yA, 7)[1]
    print(f"trajectory '{m}': clip-III recognition UAR  train-OOF {tr_u:.2f} | {EVAL_SPLIT} per-fold mean {ev_u:.2f} | "
          f"{EVAL_SPLIT} feature-avg {fa_u:.2f}")
pd.DataFrame(rec_log).to_csv(f"{OUT_DIR}/g3b_recognizers.csv", index=False)
np.savez(f"{OUT_DIR}/g3b_trajectories.npz", ev_ctx=np.array(ev_ctx),
         **{f"train_{m}_clips": np.array(list(TRAJ[m][0])) for m in TRAJ},
         **{f"train_{m}_vecs": np.stack(list(TRAJ[m][0].values())) for m in TRAJ},
         **{f"ev_{m}_fold{k}": np.stack([TRAJ[m][1][k][c] for c in ev_ctx]) for m in TRAJ for k in range(N_FOLDS)})

## Stage 2 — forecasters under both selection protocols

In [ ]:
fc_train = train_all[~train_all.source_folder.isin(inner_dev_sources)].reset_index(drop=True)
fc_dev = train_all[train_all.source_folder.isin(inner_dev_sources)].reset_index(drop=True)
COL = {1: 'clip1', 2: 'clip2', 3: 'clip3'}
_T_CACHE = {}


def make_T(rows_name, d, clips, traj_map, map_key):
    key = (rows_name, clips, map_key)
    if key not in _T_CACHE:
        vals = d[[COL[k] for k in clips]].values
        idx = torch.tensor([[CIDX[c] for c in r] for r in vals], device=DEVICE)
        if traj_map is None:
            rec = torch.zeros(len(d), len(clips), N_REC, device=DEVICE)
        else:
            rec = torch.tensor(np.stack([np.stack([traj_map[c] for c in r]) for r in vals]), device=DEVICE)
        _T_CACHE[key] = (idx, rec, torch.tensor(d.yB.values, device=DEVICE))
    return _T_CACHE[key]


def fc_predict(model, T, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(T[0]), bs):
            out.append(F.softmax(model(T[0][i:i + bs], T[1][i:i + bs]), -1).cpu())
    return torch.cat(out).numpy()


def predict_avg(model, T_list):
    """One prediction per fold-model trajectory, then average the probabilities."""
    return np.mean([fc_predict(model, T) for T in T_list], 0)


def sets_for(arm, clips, mode, protocol):
    assert clips == (1, 2, 3)[3 - len(clips):], 'clips must be a suffix of (1, 2, 3)'
    tmap, versions, favg = TRAJ[mode] if arm == 'traj' else (None, [None], None)
    mk = str(mode)
    tr_rows, tr_name = (train_all, 'train_all') if protocol == 'val' else (fc_train, 'fc_train')
    T_tr = make_T(tr_name, tr_rows, clips, tmap, mk)
    T_ev = [make_T('ev', ev, clips, v, f"{mk}_fold{k}") for k, v in enumerate(versions)]
    T_sel = T_ev if protocol == 'val' else [make_T('fc_dev', fc_dev, clips, tmap, mk)]
    T_favg = make_T('ev', ev, clips, favg, f"{mk}_favg") if arm == 'traj' else None
    return T_tr, T_sel, T_ev, T_favg


def train_forecaster(arm, clips, mode, protocol, seed):
    seed_all(seed)
    T_tr, T_sel, T_ev, T_favg = sets_for(arm, clips, mode, protocol)
    model = Forecaster(use_raw=(arm == 'B1'), use_traj=(arm == 'traj')).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    idx, rec, y = T_tr
    y_sel = T_sel[0][2].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(FC_EPOCHS):
        model.train()
        perm = torch.randperm(len(y), device=DEVICE)
        for i in range(0, len(perm), FC_BATCH):
            j = perm[i:i + FC_BATCH]
            loss = F.cross_entropy(model(idx[j], rec[j]), y[j])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sel_uar = war_uar(predict_avg(model, T_sel).argmax(1), y_sel, 7)[1]
        if sel_uar > best:
            best, bad = sel_uar, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    p = predict_avg(model, T_ev)
    p_favg = fc_predict(model, T_favg) if T_favg is not None else None
    return p, p_favg, best


yB = ev.yB.values
src = ev.source_folder.values
cert = (ev.unc_B == '1').values
results, PROBS = [], {}
for name, arm, clips, mode, protocol in EXPERIMENTS:
    PROBS[name] = []
    for seed in SEEDS:
        p, p_favg, sel = train_forecaster(arm, clips, mode, protocol, seed)
        PROBS[name].append(p)
        w, u = war_uar(p.argmax(1), yB, 7)
        wc, uc = war_uar(p[cert].argmax(1), yB[cert], 7)
        r = {'exp': name, 'protocol': protocol, 'seed': seed, 'sel_UAR': sel, 'UAR': u, 'WAR': w,
             'UAR_certain': uc, 'WAR_certain': wc}
        if p_favg is not None:
            r['WAR_featavg'], r['UAR_featavg'] = war_uar(p_favg.argmax(1), yB, 7)
        results.append(r)
        print({k: round(v, 2) if isinstance(v, float) else v for k, v in r.items()})
    torch.cuda.empty_cache()

res = pd.DataFrame(results)
res.to_csv(f"{OUT_DIR}/g3b_results_per_seed.csv", index=False)
np.savez(f"{OUT_DIR}/g3b_{EVAL_SPLIT}_probs.npz", sample_id=ev.sample_id.values,
         **{n.replace('@', '_at_').replace('-', '_'): np.stack(v) for n, v in PROBS.items()})
print("\n== mean ± std over forecaster seeds ==")
print(res.drop(columns=['seed', 'protocol']).groupby('exp', sort=False).agg(['mean', 'std']).round(2).to_string())

## Logistic regression on the trajectory (simplest forecaster)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

tmap, versions, _ = TRAJ['avg']
LR_PROBS = {}
for clips in [(3,), (2, 3), (1, 2, 3)]:
    Xt = np.hstack([np.stack([tmap[c] for c in train_all[COL[k]]]) for k in clips])
    yt = train_all.yB.values
    best = None
    for C in [0.01, 0.03, 0.1, 0.3, 1, 3]:
        s = []
        for a, b in GroupKFold(5).split(Xt, yt, train_all.source_folder):
            pred = LogisticRegression(max_iter=3000, C=C).fit(Xt[a], yt[a]).predict(Xt[b])
            s.append(war_uar(pred, yt[b], 7)[1])
        if best is None or np.mean(s) > best[0]:
            best = (np.mean(s), C)
    clf = LogisticRegression(max_iter=3000, C=best[1]).fit(Xt, yt)
    p = np.mean([clf.predict_proba(np.hstack([np.stack([v[c] for c in ev[COL[k]]]) for k in clips])) for v in versions], 0)
    name = "LR_traj_" + {(3,): "III", (2, 3): "II-III", (1, 2, 3): "I-III"}[clips]
    LR_PROBS[name] = p
    report(f"{name} (C={best[1]})", p.argmax(1), yB, src)

## Paired comparisons, per-episode wins and the gate

In [ ]:
def paired_diff_ci(pa, pb, y, src, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        wa, ua = war_uar(pa[idx], y[idx], 7)
        wb, ub = war_uar(pb[idx], y[idx], 7)
        d.append((ua - ub, wa - wb))
    return np.percentile(np.array(d), [2.5, 97.5], axis=0)


ENS = {n: np.mean(v, 0).argmax(1) for n, v in PROBS.items()}
ENS.update({n: p.argmax(1) for n, p in LR_PROBS.items()})
print(f"== seed-ensemble, {EVAL_SPLIT} (95% episode-bootstrap CI) ==")
for n in ENS:
    report(n, ENS[n], yB, src)

PAIRS = [("traj_I-III", "B1"), ("LR_traj_I-III", "B1"), ("traj_I-III@val", "B1@val"),
         ("traj_II-III", "traj_I-III"), ("traj_III", "traj_I-III"),
         ("traj_II-III@val", "traj_I-III@val"), ("traj_III@val", "traj_I-III@val")] + \
        [(f"traj_I-III_rec{r}", "traj_I-III") for r in REC_SEEDS]
print("\n== paired differences (seed-ensemble; per-seed wins where both are seeded) ==")
verdict = {}
for a, b in PAIRS:
    lo, hi = paired_diff_ci(ENS[a], ENS[b], yB, src)
    wa, ua = war_uar(ENS[a], yB, 7); wb, ub = war_uar(ENS[b], yB, 7)
    wins = ""
    if a in PROBS and b in PROBS:
        pa = res[res.exp == a].set_index('seed'); pb = res[res.exp == b].set_index('seed')
        wins = f"({int(((pa.UAR - pb.UAR) > 0).sum())}/{len(SEEDS)} seeds UAR)"
    ep_wins = sum(war_uar(ENS[a][src == s], yB[src == s], 7)[0] > war_uar(ENS[b][src == s], yB[src == s], 7)[0]
                  for s in np.unique(src))
    verdict[(a, b)] = (ua - ub, lo[0], hi[0])
    print(f"{a:<18} - {b:<15} ΔUAR {ua - ub:+5.2f} [{lo[0]:+5.2f},{hi[0]:+5.2f}]  ΔWAR {wa - wb:+5.2f} "
          f"[{lo[1]:+5.2f},{hi[1]:+5.2f}]  {wins}  episodes won (WAR) {ep_wins}/{len(np.unique(src))}")

print("\n== feature-quality mismatch diagnostic (trajectory experiments) ==")
diag = res.dropna(subset=['UAR_featavg']).groupby('exp', sort=False)[['UAR', 'UAR_featavg', 'WAR', 'WAR_featavg']].mean().round(2)
print(diag.to_string())

d_main = verdict[("traj_I-III", "B1")]
d_val = verdict[("traj_I-III@val", "B1@val")]
rec_spread = res[res.exp.str.startswith("traj_I-III_rec")].groupby('exp').UAR.mean()
print("\n== GATE ==")
print(f"1. traj_I-III vs B1 (inner_dev): ΔUAR {d_main[0]:+.2f}, CI lower bound {d_main[1]:+.2f}  -> "
      f"{'PASS' if d_main[1] > 0 else ('WEAK (positive, CI includes 0)' if d_main[0] > 0 else 'FAIL')}")
print(f"2. traj_I-III@val vs B1@val (report protocol): ΔUAR {d_val[0]:+.2f}  -> {'PASS' if d_val[0] >= 0 else 'FAIL'}")
print(f"3. recognizer-seed spread of traj_I-III UAR: {rec_spread.max() - rec_spread.min():.2f} points "
      f"-> {'PASS' if rec_spread.max() - rec_spread.min() <= 1.5 else 'UNSTABLE'}")
print("4. clip ablation: see traj_II-III / traj_III rows above (negative Δ = earlier clips help)")